# Onwordly experiment (Kaggle)

Settings → Accelerator: **GPU T4 x2**, Internet: **On**. Add-ons → Secrets: add **GITHUB_TOKEN** (fine-grained token with *Contents: read and write* on HereLiesAz/onwordly) and attach it to this notebook.

Then *Save Version → Save & Run All (Commit)*. Results are saved as the notebook's Kaggle output in `/kaggle/working/results` and, the moment the run ends (even if it fails part-way), pushed to GitHub on a new branch `kaggle-results/<timestamp>`; the run cell prints the PR link.

In [ ]:
# The clone lives outside /kaggle/working, so it never lands in the saved output
# and nothing has to delete it (deleting it broke the editable install).
%cd /tmp
![ -d /tmp/onwordly ] || git clone -q https://github.com/HereLiesAz/onwordly.git /tmp/onwordly
%cd /tmp/onwordly
!git pull -q
!pip install -q -e '.[train]'

In [ ]:
# One experiment, e.g. 008 (no manifest line: the experiment's own manifest is used),
# or a batch queued across both GPUs: 'experiment: batch\njobs: 001-suite,002,008\nseeds: 3303,4404,5505\n'.
PLAN = """experiment: 008
mode: single
"""
LABEL = 'experiment 008'
open('/kaggle/working/plan', 'w').write(PLAN)
!nvidia-smi -L

In [ ]:
import os, subprocess
from onwordly.publish import publish_results

os.chdir('/tmp/onwordly')
try:
    subprocess.run(['onwordly-kaggle', '--plan', '/kaggle/working/plan',
                    '--output-root', '/kaggle/working/results'], check=True)
finally:
    # Upload whatever exists, finished or not, as soon as the run stops.
    publish_results('/kaggle/working/results', label=LABEL)

In [ ]:
from pathlib import Path
from IPython.display import Markdown, display
for report in sorted(Path('/kaggle/working/results').rglob('RESULTS.md')):
    display(Markdown(report.read_text()))